# 02_Content-based recommender

Builds a "more like this" recommender for the anime catalog using only title content: synopsis text, genres, studio and format. Content-based recommendation is how a streaming service can recommend a title before it has any viewing history (the cold-start problem).

The dataset has no user-level ratings, so standard accuracy metrics such as Recall@K cannot be computed. The models are instead compared on properties that can be measured from content alone: genre relevance, how often they return the same franchise, catalog coverage, and how strongly they favor popular titles.

Outputs for the Streamlit app: `data/processed/catalog.parquet` and `data/processed/neighbors.parquet`.

## 1. Load catalog

In [1]:
import re

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, normalize

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

df = pd.read_parquet('../data/processed/anime_clean.parquet')
print(f"Catalog titles: {df['in_catalog'].sum():,}")

Catalog titles: 6,801


Adult titles rated `Rx - Hentai` are removed. The app is a public demo, and recommending explicit content to someone who searched for a mainstream title would be a product failure regardless of content similarity.

In [2]:
cat = df[(df['in_catalog'] == 1) & (df['rating'] != 'Rx - Hentai')].reset_index(drop=True)
print(f'Titles after removing adult content: {len(cat):,}')

Titles after removing adult content: 6,737


## 2. Feature blocks

Each block describes one aspect of a title:

| Block | Encoding | What it captures |
|---|---|---|
| `synopsis` | TF-IDF on unigrams and bigrams | Story, setting, themes, character names |
| `genres` | Multi-hot | Genre and demographic tags |
| `studios` | Multi-hot, studios with 3+ titles | Production style |
| `meta` | One-hot of type, source, age rating | Format and audience |

Synopses often end with a credit line such as `[Written by MAL Rewrite]`, which is removed so it does not make unrelated titles look similar.

In [3]:
def clean_synopsis(s):
    s = re.sub(r'\[Written by[^\]]*\]', ' ', s)
    s = re.sub(r'\(Source:[^)]*\)', ' ', s)
    return s.strip()

synopsis = cat['synopsis'].apply(clean_synopsis)
tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, ngram_range=(1, 2),
                        min_df=3, max_df=0.5)
X_syn = tfidf.fit_transform(synopsis)

X_gen = MultiLabelBinarizer(sparse_output=True).fit_transform(cat['genres'])

studio_counts = cat['studios'].explode().value_counts()
keep_studios = set(studio_counts[studio_counts >= 3].index)
studios_kept = cat['studios'].apply(lambda s: [x for x in s if x in keep_studios])
X_stu = MultiLabelBinarizer(sparse_output=True).fit_transform(studios_kept)

meta = cat[['type', 'source', 'rating']].fillna('Unknown')
X_meta = OneHotEncoder(handle_unknown='ignore').fit_transform(meta)

blocks = {'synopsis': X_syn, 'genres': X_gen, 'studios': X_stu, 'meta': X_meta}
for k, v in blocks.items():
    print(f'{k:<9} {v.shape[1]:>6,} features')

synopsis  23,601 features
genres        78 features
studios      325 features
meta          25 features


## 3. Combine blocks with explicit weights

Each block is L2-normalized and multiplied by the square root of its weight before stacking. The cosine similarity between two titles then equals a weighted average of the per-block cosine similarities, so each weight is the share of the final similarity that block controls. Without this step, the block with the most features (synopsis) would dominate by size alone.

Popularity and score are deliberately not included. In the EDA, attention was highly concentrated, and adding engagement counts as features makes every title look similar to the most popular ones.

In [4]:
WEIGHTS = {'synopsis': 0.45, 'genres': 0.35, 'studios': 0.10, 'meta': 0.10}

def build_matrix(blocks, weights):
    parts = [normalize(blocks[k]) * np.sqrt(w) for k, w in weights.items()]
    return normalize(sparse.hstack(parts).tocsr())

X = build_matrix(blocks, WEIGHTS)
print(X.shape)

(6737, 24029)


## 4. Candidate neighbors

For every title, the 50 most similar titles by cosine similarity are retrieved once. All recommenders below re-rank or filter this same candidate pool, so differences between them come only from the ranking rule.

In [5]:
N_CAND = 50
nn = NearestNeighbors(n_neighbors=N_CAND + 1, metric='cosine', algorithm='brute').fit(X)
dist, idx = nn.kneighbors(X)
sim = 1 - dist

# Drop each title from its own neighbor list
cand_idx, cand_sim = [], []
for i in range(len(cat)):
    keep = idx[i] != i
    cand_idx.append(idx[i][keep][:N_CAND])
    cand_sim.append(sim[i][keep][:N_CAND])
cand_idx, cand_sim = np.array(cand_idx), np.array(cand_sim)
print(cand_idx.shape)

(6737, 50)


## 5. Recommenders compared

| Model | Rule |
|---|---|
| Popularity baseline | The 5 most-watched titles in the catalog, the same for every user |
| Content | Top 5 candidates by content similarity |
| Content, new franchise | Same, but skips titles from the query's own franchise |
| Content + popularity boost | Similarity multiplied by a popularity factor, `sim × pop^α` |

Franchise is approximated from titles: two titles are treated as the same franchise if their first two title words match (or the first word, when a title has only one). This catches sequels and movies such as *Naruto* and *Naruto: Shippuuden*, while keeping *One Piece* and *One Punch Man* apart. It is a heuristic and will miss franchises whose sequels change name.

The popularity factor `pop` is log members scaled to 0–1. α > 0 favors popular titles; α < 0 favors niche ones.

In [6]:
def title_tokens(name):
    return re.findall(r'[a-z0-9]+', str(name).lower())

tokens = cat['name'].apply(title_tokens).tolist()

def same_franchise(i, j):
    a, b = tokens[i], tokens[j]
    k = min(2, len(a), len(b))
    return k > 0 and a[:k] == b[:k]

log_m = np.log10(cat['members'].clip(lower=1).to_numpy())
pop = (log_m - log_m.min()) / (log_m.max() - log_m.min())

K = 5

def recommend(i, alpha=0.0, new_franchise=False, k=K):
    c_idx, c_sim = cand_idx[i], cand_sim[i]
    if new_franchise:
        mask = np.array([not same_franchise(i, j) for j in c_idx])
        c_idx, c_sim = c_idx[mask], c_sim[mask]
    score = c_sim * np.power(pop[c_idx] + 1e-6, alpha)
    order = np.argsort(-score)[:k]
    return c_idx[order], c_sim[order]

top_popular = np.argsort(-cat['members'].to_numpy())[:K + 1]

def recommend_popular(i, k=K):
    return np.array([j for j in top_popular if j != i][:k]), None

## 6. Evaluation

Without user ratings, each model is scored on four properties, averaged over every title in the catalog used as a query:

- **Genre Jaccard**: overlap between the query's genres and each recommendation's genres. Higher means more relevant.
- **Same-franchise rate**: share of recommendations from the query's own franchise. A viewer who just finished a show usually already knows about its sequels, so high values mean less discovery.
- **Catalog coverage**: share of catalog titles recommended at least once. Low coverage means most of the catalog is never surfaced.
- **Popular share**: share of recommendations from the top 10% most-watched titles. The catalog's own rate is 10%.

In [7]:
genre_sets = [set(g) for g in cat['genres']]
top10_cut = np.quantile(cat['members'], 0.90)
is_top10 = (cat['members'] >= top10_cut).to_numpy()

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 0.0

def evaluate(rec_fn):
    jac, fran, top, seen = [], [], [], set()
    for i in range(len(cat)):
        recs, _ = rec_fn(i)
        for j in recs:
            jac.append(jaccard(genre_sets[i], genre_sets[j]))
            fran.append(same_franchise(i, j))
            top.append(is_top10[j])
            seen.add(j)
    return {'genre_jaccard': np.mean(jac), 'same_franchise_rate': np.mean(fran),
            'catalog_coverage': len(seen) / len(cat), 'popular_share': np.mean(top)}

models = {
    'Popularity baseline':        recommend_popular,
    'Content':                    lambda i: recommend(i),
    'Content, new franchise':     lambda i: recommend(i, new_franchise=True),
    'New franchise, niche (α=-1)':   lambda i: recommend(i, alpha=-1, new_franchise=True),
    'New franchise, popular (α=1)':  lambda i: recommend(i, alpha=1, new_franchise=True),
    'New franchise, popular (α=3)':  lambda i: recommend(i, alpha=3, new_franchise=True),
}

results = pd.DataFrame({name: evaluate(fn) for name, fn in models.items()}).T
results.round(3)

,genre_jaccard,same_franchise_rate,catalog_coverage,popular_share
Popularity baseline,0.098,0.001,0.001,1.000
Content,0.759,0.240,0.948,0.120
"Content, new franchise",0.706,0.000,0.904,0.109
"New franchise, niche (α=-1)",0.604,0.000,0.373,0.000
"New franchise, popular (α=1)",0.608,0.000,0.392,0.539
"New franchise, popular (α=3)",0.584,0.000,0.344,0.583


## 7. Example recommendations

Side-by-side top 5 for a few well-known titles across different genres, comparing plain content similarity with the franchise-filtered version.

In [8]:
name_to_idx = {n: i for i, n in enumerate(cat['name'])}

def show(name, **kwargs):
    if name not in name_to_idx:
        print(f'{name}: not in catalog')
        return None
    i = name_to_idx[name]
    recs, sims = recommend(i, **kwargs)
    out = cat.loc[recs, ['name', 'year', 'type', 'score', 'members']].copy()
    out['similarity'] = np.round(sims, 3)
    return out.reset_index(drop=True)

examples = ['Death Note', 'Hunter x Hunter (2011)', 'K-On!', 'Spy x Family', 'Mushishi']
for name in examples:
    print(f'\n=== {name} | content ===')
    display(show(name))
    print(f'=== {name} | content, new franchise ===')
    display(show(name, new_franchise=True))


=== Death Note | content ===


,name,year,type,score,members,similarity
0,ChäoS;HEAd,2008.0,TV,6.28,300187,0.415
1,Mouryou no Hako,2008.0,TV,7.14,68401,0.389
2,AI no Idenshi,2023.0,TV,6.52,51146,0.388
3,Platinum End,2021.0,TV,6.01,423875,0.381
4,Boogiepop wa Warawanai (2019),2019.0,TV,7.07,246779,0.376


=== Death Note | content, new franchise ===


,name,year,type,score,members,similarity
0,ChäoS;HEAd,2008.0,TV,6.28,300187,0.415
1,Mouryou no Hako,2008.0,TV,7.14,68401,0.389
2,AI no Idenshi,2023.0,TV,6.52,51146,0.388
3,Platinum End,2021.0,TV,6.01,423875,0.381
4,Boogiepop wa Warawanai (2019),2019.0,TV,7.07,246779,0.376



=== Hunter x Hunter (2011) | content ===


,name,year,type,score,members,similarity
0,Hunter x Hunter Movie 1: Phantom Rouge,2013.0,Movie,7.29,222684,0.637
1,Hunter x Hunter,1999.0,TV,8.44,658062,0.621
2,Hunter x Hunter Movie 2: The Last Mission,2013.0,Movie,7.29,203008,0.515
3,Claymore,2007.0,TV,7.73,732352,0.483
4,Sousou no Frieren 2nd Season,2026.0,TV,8.84,618902,0.468


... (10 outputs left out)


=== Mushishi | content ===


,name,year,type,score,members,similarity
0,Mushishi Zoku Shou,2014.0,TV,8.68,330849,0.704
1,Mushishi Zoku Shou 2nd Season,2014.0,TV,8.72,296675,0.659
2,Mushishi Zoku Shou: Suzu no Shizuku,2015.0,Movie,8.58,142734,0.593
3,Zatsu Tabi: That's Journey,2025.0,TV,6.93,41109,0.349
4,Mokke,2007.0,TV,7.17,17681,0.346


=== Mushishi | content, new franchise ===


,name,year,type,score,members,similarity
0,Zatsu Tabi: That's Journey,2025.0,TV,6.93,41109,0.349
1,Mokke,2007.0,TV,7.17,17681,0.346
2,Master Keaton,1998.0,TV,7.66,33447,0.325
3,Hakushaku to Yousei,2008.0,TV,7.19,135167,0.324
4,Kami no Niwatsuki Kusunoki-tei,2026.0,TV,7.16,40087,0.319


## 8. Export for the app

The app uses the franchise-filtered content model, which gives a viewer new titles to watch rather than sequels of the show they searched for. `FINAL_ALPHA` can be changed after reviewing the evaluation table; 0 means pure content similarity. The top 10 recommendations per title are saved so the app only needs to look them up.

In [9]:
FINAL_ALPHA = 0.0
FINAL_NEW_FRANCHISE = True
TOP_N = 10

ids = cat['anime_id'].to_numpy()
rows = []
for i in range(len(cat)):
    recs, sims = recommend(i, alpha=FINAL_ALPHA, new_franchise=FINAL_NEW_FRANCHISE, k=TOP_N)
    for r, (j, s) in enumerate(zip(recs, sims), start=1):
        rows.append((ids[i], ids[j], r, float(s)))

neighbors = pd.DataFrame(rows, columns=['source_id', 'rec_id', 'rank', 'similarity'])

catalog = cat[['anime_id', 'name', 'english_name', 'year', 'type', 'score', 'members', 'image_url']].copy()
catalog['genres'] = cat['genres'].apply(lambda g: ', '.join(g))

neighbors.to_parquet('../data/processed/neighbors.parquet', index=False)
catalog.to_parquet('../data/processed/catalog.parquet', index=False)
print(neighbors.shape, catalog.shape)

(67353, 4) (6737, 9)


## Key findings

- **The catalog used for recommendations has 6,737 titles** after removing 64 adult titles. The combined feature matrix has 24,029 features, of which 23,601 come from the synopsis.
- **Content similarity is far more relevant than popularity.** Content recommendations share 0.759 genre Jaccard with the query, versus 0.098 for the popularity baseline, and they surface 94.8% of the catalog compared with 0.1%.
- **Plain content similarity mostly returns sequels.** 24.0% of its recommendations come from the query's own franchise; for *K-On!*, *Spy x Family* and *Mushishi*, most of the top 5 are later seasons or movies of the same show. Filtering out the query's franchise removes this entirely at a small cost: genre Jaccard falls from 0.759 to 0.706 and coverage from 94.8% to 90.4%.
- **The franchise-filtered recommendations are recognizable and plausible.** *K-On!* returns other school-club comedies (*Lucky☆Star*, *Yuyushiki*, *Bocchi the Rock!*); *Hunter x Hunter (2011)* returns adventure shounen (*Claymore*, *Magi*, *Frieren*); *Death Note* returns psychological thrillers including *Platinum End*, from the same creators.
- **Popularity re-ranking hurts in both directions.** Boosting popular titles (α=1) raises the popular share from 10.9% to 53.9% but cuts genre Jaccard to 0.608 and coverage to 39.2%. Boosting niche titles (α=-1) also cuts coverage, to 37.3%, because the re-ranking concentrates recommendations on whichever end of the popularity scale it favors. The app therefore uses pure content similarity (α=0) with the franchise filter, whose popular share (10.9%) matches the catalog's own 10%.
- **Limitations.** The franchise rule is based on title words, so it misses sequels whose names change. A single list can also contain two entries from another franchise (*Magi* twice for *Hunter x Hunter*), or a later season instead of the first (*Frieren 2nd Season*). Without user ratings, relevance is measured through genre overlap, which is a proxy rather than a measure of what viewers actually enjoy.